In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedGroupKFold

vitals = [
    'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 
    'HR_Delta', 'SBP_Delta', 'Temp_Delta', 'Resp_Delta'
]
labs = [
    'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 
    'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 
    'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 
    'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 
    'Fibrinogen', 'Platelets'
]
demographics = [
    'Age', 'Gender', 'ICULOS', 'high_oxygen_support', 'baseline_risk_index'
]
scores = ['NEWS', 'SOFA']

In [2]:
def calculate_news_score(row):
    # Calculates standard NEWS2 score based on RCP guidelines.
    score = 0
    # Respiration Rate
    if row['Resp'] <= 8 or row['Resp'] >= 25: score += 3
    elif 21 <= row['Resp'] <= 24: score += 2
    elif 9 <= row['Resp'] <= 11: score += 1
    # SpO2
    if row['O2Sat'] <= 91: score += 3
    elif 92 <= row['O2Sat'] <= 93: score += 2
    elif 94 <= row['O2Sat'] <= 95: score += 1
    # Temperature (Celsius)
    if row['Temp'] <= 35.0: score += 3
    elif row['Temp'] >= 39.1: score += 2
    elif (35.1 <= row['Temp'] <= 36.0) or (38.1 <= row['Temp'] <= 39.0): score += 1
    # Systolic BP
    if row['SBP'] <= 90 or row['SBP'] >= 220: score += 3
    elif 91 <= row['SBP'] <= 100: score += 2
    elif 101 <= row['SBP'] <= 110: score += 1
    # Heart Rate
    if row['HR'] <= 40 or row['HR'] >= 131: score += 3
    elif 111 <= row['HR'] <= 130: score += 2
    elif (41 <= row['HR'] <= 50) or (91 <= row['HR'] <= 110): score += 1
    return score

def calculate_sofa_score(row):
    # Calculates partial SOFA score based on clinical components.
    score = 0
    # Platelets (Coagulation)
    if row['Platelets'] < 20: score += 4
    elif row['Platelets'] < 50: score += 3
    elif row['Platelets'] < 100: score += 2
    elif row['Platelets'] < 150: score += 1
    # Creatinine (Renal)
    if row['Creatinine'] >= 5.0: score += 4
    elif 3.5 <= row['Creatinine'] <= 4.9: score += 3
    elif 2.0 <= row['Creatinine'] <= 3.4: score += 2
    elif 1.2 <= row['Creatinine'] <= 1.9: score += 1
    # Bilirubin (Liver)
    if 'Bilirubin_total' in row and not pd.isna(row['Bilirubin_total']):
        if row['Bilirubin_total'] >= 12.0: score += 4
        elif 6.0 <= row['Bilirubin_total'] <= 11.9: score += 3
        elif 2.0 <= row['Bilirubin_total'] <= 5.9: score += 2
        elif 1.2 <= row['Bilirubin_total'] <= 1.9: score += 1
    return score

In [3]:
# Load raw data
df_raw = pd.read_csv(r'D:\Uni\Year_3_sem_2\XAI\Project\phase_2\Transformer_Engineered_Dataset.csv')

print("Calculating clinical scores...")
df_raw['NEWS'] = df_raw.apply(calculate_news_score, axis=1)
df_raw['SOFA'] = df_raw.apply(calculate_sofa_score, axis=1)

# Identify which specified features are actually present
available_features = [f for f in vitals + labs + demographics + scores if f in df_raw.columns]
df_filtered = df_raw[['Patient_ID', 'SepsisLabel'] + available_features].copy()

Calculating clinical scores...


In [4]:
def aggregate_to_transformer_steps(df):
    df['bucket'] = (df['ICULOS'] - 1) // 4
    # Mean for continuous vitals/scores, Last for labs
    agg_map = {feat: 'mean' if feat in vitals + scores else 'last' for feat in available_features}
    agg_map['SepsisLabel'] = 'max'
    
    def process_patient(group):
        # Force exactly 6 buckets (first 24 hours)
        res = group.groupby('bucket').agg(agg_map).reindex(range(6))
        # Forward fill then backward fill within the patient window
        return res.ffill().bfill() 

    return df.groupby('Patient_ID').apply(process_patient).reset_index()

print("Bucketing into 4-hour intervals...")
df_bucketed = aggregate_to_transformer_steps(df_filtered)

Bucketing into 4-hour intervals...


C:\Users\nour1\AppData\Local\Temp\ipykernel_33436\3397604833.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('Patient_ID').apply(process_patient).reset_index()


In [5]:
# 1. Identify unique patients and clean labels
patient_ids = df_bucketed['Patient_ID'].unique()
labels_series = df_bucketed.groupby('Patient_ID')['SepsisLabel'].max().fillna(0)
labels = labels_series.astype(int).values

# 2. Stratified Group K-Fold split
sgkf = StratifiedGroupKFold(n_splits=5)
train_idx, val_idx = next(sgkf.split(patient_ids, labels, groups=patient_ids))
train_pids, val_pids = patient_ids[train_idx], patient_ids[val_idx]

# 3. Imputation (Training Median only to avoid leakage)
train_data = df_bucketed[df_bucketed['Patient_ID'].isin(train_pids)]
train_median = train_data[available_features].median()
df_bucketed[available_features] = df_bucketed[available_features].fillna(train_median)

# 4. Standard Scaling
scaler = StandardScaler()
df_bucketed[available_features] = scaler.fit_transform(df_bucketed[available_features])

In [ ]:
def build_tensor(df, pids):
    subset = df[df['Patient_ID'].isin(pids)]
    X = subset[available_features].values.reshape(len(pids), 6, len(available_features))
    y = subset.groupby('Patient_ID')['SepsisLabel'].max().fillna(0).astype(int).values
    return X, y

# Generate the base tensors
X_train, y_train = build_tensor(df_bucketed, train_pids)
X_val, y_val = build_tensor(df_bucketed, val_pids)

# SMOTE Application (Training Data Only)
print(f"Applying SMOTE... Original Class Distribution: {np.bincount(y_train)}")

# 1. Flatten X from (Samples, 6, 45) to (Samples, 6 * 45) for SMOTE
n_samples, n_timesteps, n_features = X_train.shape
X_train_flattened = X_train.reshape(n_samples, n_timesteps * n_features)

# 2. Apply SMOTE to the flattened training data
smote = SMOTE(random_state=42)
X_train_resampled_flat, y_train_resampled = smote.fit_resample(X_train_flattened, y_train)

# 3. Reshape back to the 3D tensor format for the Transformer
X_train_resampled = X_train_resampled_flat.reshape(-1, n_timesteps, n_features)

print(f"SMOTE Complete. New Class Distribution: {np.bincount(y_train_resampled)}")

output_dir = r'D:\Uni\Year_3_sem_2\XAI\Project\phase_2\transformer_processed_data'
os.makedirs(output_dir, exist_ok=True)

# Save Resampled Training Data
train_path = os.path.join(output_dir, 'transformer_train_data.pkl')
with open(train_path, 'wb') as f:
    pickle.dump({
        'X': X_train_resampled, 
        'y': y_train_resampled, 
        'features': available_features
    }, f)

# Save Original Validation/Test Data (NEVER apply SMOTE to validation/test)
test_path = os.path.join(output_dir, 'transformer_test_data.pkl')
with open(test_path, 'wb') as f:
    pickle.dump({
        'X': X_val, 
        'y': y_val, 
        'features': available_features
    }, f)

print(f"Success! Data saved separately.")
print(f"Train File: {train_path} | New Shape: {X_train_resampled.shape}")
print(f"Test File: {test_path} | Shape: {X_val.shape}")

Applying SMOTE... Original Class Distribution: [24673   219]
SMOTE Complete. New Class Distribution: [24673 24673]
Success! Data saved separately.
Train File: D:\Uni\Year_3_sem_2\XAI\Project\phase_2\transformer_processed_data\transformer_train_data.pkl | New Shape: (49346, 6, 45)
Test File: D:\Uni\Year_3_sem_2\XAI\Project\phase_2\transformer_processed_data\transformer_test_data.pkl | Shape: (6224, 6, 45)
